# Seed


In [ ]:
"""
Seed script — populates the DB with synthetic transaction data
for demo/development purposes.
Run with: python -m app.seed
"""
import asyncio
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from datetime import datetime, timedelta
import uuid
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from app.config import get_settings
from app.models.user import User, Transaction
from app.models.user import TransactionType
from app.database import Base
from app.services.auth_service import hash_password
from ml.src.synthetic_data import generate_transactions, inject_anomalies

settings = get_settings()


In [ ]:
def seed():
    engine = create_engine(settings.DATABASE_URL_SYNC)
    Base.metadata.create_all(engine)
    Session = sessionmaker(bind=engine)
    session = Session()

    print("Seeding database...")

    # Create demo user
    demo_user = User(
        email="demo@expenseai.com",
        hashed_password=hash_password("demo1234"),
        full_name="Demo User",
        monthly_income=5500.0,
        monthly_savings_goal=1100.0,
        risk_tolerance="moderate",
    )
    session.add(demo_user)
    session.commit()
    session.refresh(demo_user)
    print(f"Created demo user: {demo_user.email} (password: demo1234)")

    # Generate synthetic transactions
    print("Generating synthetic transactions...")
    df = generate_transactions(n_transactions=5000, n_users=1, monthly_income=demo_user.monthly_income)
    df = inject_anomalies(df, anomaly_rate=0.02)
    df["transaction_date"] = pd.to_datetime(df["transaction_date"])

    # Assign correct type
    type_map = {"income": TransactionType.INCOME, "expense": TransactionType.EXPENSE}
    for _, row in df.iterrows():
        tx = Transaction(
            user_id=demo_user.id,
            amount=row["amount"],
            transaction_type=type_map[row["transaction_type"]],
            description=row["description"],
            merchant=row["merchant"],
            category=row["category"],
            predicted_category=row["category"],
            is_anomaly=row.get("is_anomaly", False),
            anomaly_score=row.get("anomaly_score", None),
            transaction_date=row["transaction_date"],
        )
        session.add(tx)

    session.commit()
    print(f"Inserted {len(df)} transactions")
    session.close()
    print("Done!")


if __name__ == "__main__":
    seed()
